# Dunnhumby anchored M2 체크포인트 진단 (재학습 없음)

기존 `joint_nv_anchored` seed 42 validation 체크포인트를 읽어 `ID-only`, `ID+N`, `ID+V`, `full`을 분해 평가합니다. 학습은 시작하지 않으며 test와 holdout은 열지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = '2791aabe13a65290ee2ce4b25524e7e50018f48d'
repo = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)
], check=True)
os.chdir(repo)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
assert subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip() == REVIEWED_SHA
print('code:', REVIEWED_SHA)


In [ ]:
import importlib, json, sys, torch
for module_name, module in list(sys.modules.items()):
    module_path = getattr(module, '__file__', '') or ''
    if module_path and str(repo) in str(module_path):
        del sys.modules[module_name]
importlib.invalidate_caches()

from IPython.display import display
import pandas as pd
from lightgcn_clv_joint_nv import (
    configure_anchored_dunnhumby_run,
    preflight_summary,
    run_checkpoint_diagnostics,
)
assert torch.cuda.is_available(), (
    '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
cfg = configure_anchored_dunnhumby_run()
summary = preflight_summary(cfg)
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['dataset'] == 'dunnhumby'
assert summary['seed'] == 42
assert summary['models'] == ['m1', 'joint_nv_anchored']
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
print('설정 확인 완료. 다음 셀은 기존 체크포인트를 읽어 평가만 합니다.')


In [ ]:
diagnostic_df = run_checkpoint_diagnostics(cfg)
assert diagnostic_df.attrs['training_performed'] is False

metrics = [
    'view', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
    'arp@10', 'coverage@10', 'n_distinct@10', 'eff_catalog@10',
]
print('===== 블록 분해 절대지표 =====')
display(diagnostic_df[[c for c in metrics if c in diagnostic_df.columns]])

comparison = diagnostic_df.attrs['block_comparison']
focus_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
]
print('===== M1 대비 ID 회복과 N/V 증분 =====')
display(comparison[comparison['metric'].isin(focus_metrics)].reset_index(drop=True))

print('===== 블록별 실효강도 =====')
print(json.dumps(diagnostic_df.attrs['block_score_summary'], ensure_ascii=False, indent=2))
print('===== N/V 축 분포 =====')
print(json.dumps(diagnostic_df.attrs['axis_distribution'], ensure_ascii=False, indent=2))
print('결과 파일:', diagnostic_df.attrs['result_paths'])
print('완료. 위 세 부분을 그대로 공유해 주세요.')
